In [2]:
import os
from openai import OpenAI
from src.GraphRAG import GraphRAG
from src.Chunker import Chunker
from src.GraphBuilder import GraphBuilder
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cuda:3"
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [4]:

embeddings = model.encode(
    'Hello world',
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters in the model: {num_params:,}")

Number of parameters in the model: 567,754,752


In [ ]:
texts = [doc.page_content for doc in docs_processed]

embeddings = model.encode(
    texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)


index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

In [19]:
client = OpenAI(
    base_url=os.environ['OPENROUTER_BASE_URL'], 
    api_key=os.environ["OPENROUTER_API_KEY"]
)
response = client.chat.completions.create(
    model="openrouter/free", temperature=0, response_format={"type": "text"},
    messages=[{"role": "user", "content": "Who are you"}],
).choices[0].message.content

In [ ]:
response

"I'm an autonomous software agent designed to assist with a wide range of tasks. I can help answer questions, provide information, assist with problem-solving, and engage in conversation on various topics. My purpose is to be a helpful and reliable tool for users like you."

In [13]:
os.makedirs('checkpoints', exist_ok=True)

client = OpenAI(
    base_url=os.environ['OPENROUTER_BASE_URL'], 
    api_key=os.environ["OPENROUTER_API_KEY"]
)
chunker = Chunker(
    chunk_size=512, overlap=50, encoding_model="cl100k_base",
    embed_client=client, #embed_model="google/gemini-embedding-001",
    embed_model="nomic-ai/nomic-embed-text-v1.5",
)

chunks_path = 'checkpoints/chunks.json'
if os.path.exists(chunks_path):
    chunks = chunker.load_chunks(chunks_path)
else:
    chunks = chunker.chunk_file(file_path='data/book.txt')
    chunker.save_chunks(chunks=chunks, file_path=chunks_path)

Chunking text from source: data/book.txt: 100%|██████████| 100/100 [00:00<00:00, 31131.18it/s]
Embed model: nomic-ai/nomic-embed-text-v1.5, batch size: 16:   0%|          | 0/7 [00:00<?, ?it/s]

Embed model: nomic-ai/nomic-embed-text-v1.5, batch size: 16:   0%|          | 0/7 [00:00<?, ?it/s]


BadRequestError: Error code: 400 - {'error': {'message': 'Model nomic-ai/nomic-embed-text-v1.5 does not exist', 'code': 400}}

In [18]:
import litellm, os
response = litellm.embedding(
    api_key=os.environ["OPENROUTER_API_KEY"],
    model="openrouter/openai/text-embedding-3-small",
    input=["Hello world"]
)


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



APIError: litellm.APIError: APIError: OpenrouterException - {"error":{"message":"Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits","code":402}}

In [26]:
import os
import litellm

response = litellm.embedding(
    api_key=os.environ["OPENROUTER_API_KEY"],
    model="text-embedding-3-small",   # OpenAI embedding 模型
    input=["Hello world", "Test text"],
    encoding_format="float"           # 必须指定 "float" 或 "base64"
)

print(response)


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



NotFoundError: litellm.NotFoundError: NotFoundError: OpenAIException - 

In [ ]:
chunker = Chunker(chunk_size=512, overlap=50, encoding_model="cl100k_base")
os.makedirs('checkpoints', exist_ok=True)
chunks_path = 'checkpoints/chunks.json'

if os.path.exists(chunks_path):
    chunks = chunker.load_chunks(chunks_path)
else:
    chunks = chunker.chunk_file(file_path='data/book.txt')
    chunker.save_chunks(chunks_path, chunks=chunks)

In [2]:
client = OpenAI(api_key=os.environ['OPENROUTER_API_KEY'])
graph_builder = GraphBuilder(client=client)
graph_path = 'checkpoints/graph.json'
if os.path.exists(graph_path):
    graph = graph_builder.load_graph(graph_path)
else:
    graph = graph_builder.build_graph_with_chunks(chunks)
    graph_builder.save_graph(graph_path, graph=graph)

Building knowledge graph:  17%|█▋        | 17/100 [07:43<37:36, 27.19s/it]

Building knowledge graph:  20%|██        | 20/100 [12:06<1:49:17, 81.97s/it]

Error in entity extraction: Expecting ':' delimiter: line 53613 column 1 (char 71159)


Building knowledge graph:  25%|██▌       | 25/100 [14:25<47:17, 37.84s/it]  

Building knowledge graph:  31%|███       | 31/100 [20:00<1:36:35, 84.00s/it]

Error in entity extraction: Expecting ':' delimiter: line 34987 column 1 (char 55356)


Building knowledge graph:  33%|███▎      | 33/100 [23:46<2:00:32, 107.95s/it]

Error in entity extraction: Expecting ':' delimiter: line 20045 column 3 (char 40212)


Building knowledge graph:  36%|███▌      | 36/100 [39:31<1:10:15, 65.86s/it] 


KeyboardInterrupt: 

In [5]:
graph_builder.extract_nodes_and_edges_from_chunk(chunks[16])

([{'name': 'Scrooge', 'type': 'person'},
  {'name': 'Marley', 'type': 'person'},
  {'name': "Marley's Ghost", 'type': 'person'},
  {'name': 'bell', 'type': 'concept'},
  {'name': "wine-merchant's cellar", 'type': 'location'},
  {'name': 'ghosts', 'type': 'concept'},
  {'name': 'chain', 'type': 'concept'}],
 [{'source': 'Scoroge', 'target': 'Marley', 'relation': 'mentions'},
  {'source': 'Scoroge',
   'target': 'Marile',
   'relation': '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0'}])

In [6]:
graph_builder.update_graph_with_chunk(chunks[16])

In [3]:
graphRAG = GraphRAG(client=client, graph=graph, chunks=chunks)
answer = graphRAG.query("Who is Scrooge and what are his main relationships?")

Extracted entities from query:
- Scrooge
Prompt:
 Please answer the question based on the information extracted from the knowledge graph.

        Graph context information:
        Relevant Entities:
- Entity: strange voice, type: entity, occurrences: 0
- Entity: Cab, type: object, occurrences: 0
- Entity: Turkey, type: object, occurrences: 0
- Entity: Prize turkey, type: object, occurrences: 0
- Entity: Yard, type: location, occurrences: 0
- Entity: Latent moral, type: concept, occurrences: 0
- Entity: boy, type: concept, occurrences: 0
- Entity: Singer, type: person, occurrences: 0
- Entity: Spectre, type: supernatural being, occurrences: 0
- Entity: Dream, type: concept, occurrences: 0
- Entity: words, type: concept, occurrences: 0
- Entity: Phantom, type: concept, occurrences: 0
- Entity: Avarice, type: concept, occurrences: 0
- Entity: Newspapers, type: object, occurrences: 0
- Entity: Ebenezer, type: person, occurrences: 0
- Entity: sticking-plaster, type: object, occurrences: 0

In [4]:
with open('answer.md', 'w', encoding='utf-8') as f:
    f.write(answer)

In [ ]:
import os
import litellm

#response = litellm.embedding(
    api_key=os.environ["OPENROUTER_API_KEY"],
    model="text-embedding-3-small",   # OpenAI embedding 模型
    input=["Hello world", "Test text"],
    encoding_format="float"           # 必须指定 "float" 或 "base64"
)

print(response)

